# Regular expressions in practice

The regular expressions that we've related to Finite State machines are small and elegant.  That is, our development focuses on things that are easy to understand, and to prove things about. But regular expressions are also widely used in practice. The languages used today include many additions and extensions to the theoretical ideas we have seen.  To disinguish the two kinds, we will describe the ones used in practice as *regexes*. 

## The basics

We've defined a regular expression, and now a regex, as a pattern that describes a set of strings.
We say that strings in the set *match* the regex.

An example is that if we are looking through a file of source code for looping constructs we might give the editor the regex
`(for)|(while)|(unless)`.
Recall that the vertical bars mean alternation, that is, "or", and the parentheses group.

Another example is that if we are looking for strings of at least one minus sign, we could give the 
editor the expression `--*`.
The Kleene star `*` matches repetition.

### Exercises

1. Name the three strings that match `(for)|(while)|(unless)`.

2. Give a regular expression matching strings that begin with a lower-case ASCII letter, then include
at least two digits, and then end in an upper case letter.

3. Which binds more tightly: repetition or concatenation? 
Which binds more tightly: concatenation or alternation?


## Racket syntax for regexes

A Racket regular expression is a string prefixed with `#rx` or `#px`.
The second includes some cinvenient extensions so we will use it, as in `#px"abc"`.

Test for matches with `regexp-match`.

In [2]:
(regexp-match #px"a(b|c)d" "xyz")

#f

In [3]:
(regexp-match #px"a(b|c)d" "acd")

'("acd" "c")

If the string does not match then Racket returns `#f`.  If the string does match then Racket returns a list. 
(Note that a list is not `#f`, so we can use the construct `(if (regexp-match #px"a(b|)d" "acd") ...)`.)

In a success list, the first string is the part of the input matching the regex. If there are two or more matching substrings then it shows the earlier one.
The other items in the list are strings matching parenthesized sub-expressions. 
(These come in order of the open parenthesis. Matches for the sub-expressions are provided in the order of the opening parentheses in the regex. If the overall regex can succeed without a match for the sub-expression, then instead of a substring, you see a `#f`.)

In [10]:
(regexp-match #px"a(b|(c|de))f" "adef")

'("adef" "de" "de")

Here, the `de` is what matches for both the first and second open parenthesis.

The empty string is perfectly legal here.

In [4]:
(regexp-match #px"(a||b)c" "c")

'("c" "")

### Anchors

A very important practial point about regexes, in contrast with the theoretical regular expressions,
is that they are not anchored at the start of the string.
Instead, the string matches if the regex pattern is anywhere in that string.

In [5]:
(regexp-match #px"ab" "cabd")

'("ab")

This is what, in practice, users usually want.

If you want to instead anchor the searching to the string start, put a caret, a `^`, at the start of the regex.

In [12]:
(regexp-match #px"^ab" "cabd")

#f

In [13]:
(regexp-match #px"^ca" "cabd")

'("ca")

There is a similar way to make the search extend all the way to the end of the string.
Put a dollar sign at the end of the regex. 

In [14]:
(regexp-match #px"ca" "bcad")

'("ca")

In [16]:
(regexp-match #px"ca$" "bcad")

#f

In [17]:
(regexp-match #px"^c(a|b)d$" "cbd")

'("cbd" "b")

## Sets

In the book's development of theoretical regular expressions, the alphabets had two
or three characters.
An alphabet in practice usually includes at least
include at least ASCII’s printable characters such as a-z, A-Z, 0-9, space, etc. 
And often in practice the alphabet contains
all of Unicode’s more than one hundred thousand
characters.
We need manageable ways to describe
these sets of characters.

The first tool is square brackets.
Put a list of characters between the brackets and the string will match if any of those characters works.

In [20]:
(regexp-match #px"m[aiu]x" "mix")

'("mix")

You can call for a sequence (the interval includes all characters with Unicode code points in that range).

In [18]:
(regexp-match #px"m[a-z]r" "mar")

'("mar")

In [21]:
(regexp-match #px"m[a-zA-z0-9]r" "mAr")

'("mAr")

To invert the set of matched characters, put a caret, `^ `, as the first thing inside
the bracket (and note that it is a metacharacter).
Thus, `[^0-9]` matches a non-digit
and `[^A-Za-z]` matches a character that is not an ASCII letter.

### Abbreviations

You can also use lists of abbreviations built into the system.
The next regex uses backslash-w, `\w`, to abbreviate the characters that from ordinary words, `[a-zA-Z0-9_]`
(here, "word" means that in many older programming languages, identifiers used
letters, digits, and underscores).

In [1]:
(regexp-match #px"m\\wr" "msr")

'("msr")

Note that we must double the backslash, as with `\\w`.
This does not have to do with regexes, it is about the syntax of Racket strings - that's just how we include a backslash character in a string.

In [6]:
(display "m\\wr")

m\wr

All these abbreviations are convenient:

1. `\d` for the digits, and `\D` for anything that is not a digit
2. `\w` for a word character, and `\W` for anything not in that list
3. `\s` for a whitespace character, meaning space, tab, newline, formfeed, or line return, and
`\W` for any non-whitespace character.

For example, in the twelve hour time format some typical times strings are "8:05 am"
or "10:15 pm".
This is a reasonable regex (note the empty string at the start, after the opening parenthesis).
(|0|1)\d:\d\d\s(am|pm)

In [6]:
(regexp-match #px"(|0|1)2" "2")

'("2" "")

In [8]:
(regexp-match #px"(|0|1)\\d:\\d\\d\\s(am|pm)" "1:15 am")

'("1:15 am" "" "am")

### Dot

After sets of characters, the next natural need is to match and character.
As with regular expressions, for regexes the dot `.`
matches any member of the alphabet at all.

In [9]:
(regexp-match #px"m.t" "met")

'("met")

In practice, programming languages by default have the dot match any character except newline.
They also have a way to make it match newline.

## Metacharacters

Recall that in a regular expression such as `a(b|c)d` the parentheses and the pipe
are not there to be matched. They are *metacharacters*, part of the syntax of the
regular expression. 
To match a metacharacter, prefix it with a back-
slash, `\`. 
Thus, to look for the string "(Note" put a
backslash before the open parentheses `\(Note`. 


In [ ]:
(regexp-match #px"\\(Note" "text.  (Note; we must")

Similarly, `\|` matches a pipe and `\[` matches an open square bracket. 
Match backslash itself with `\\`.
This is called *escaping* the metacharacter. 
The method described above for abbreviating sets with \d , \D , etc. is an extension of escaping.


## Quantifiers

In the theoretical cases we saw earlier, to match "at most one a" we
used $\varepsilon$|a$. 
In practice we can write something like `(|a)`, as we did above for the
times. 
But depicting the empty string by just putting nothing there
can be confusing. 
Modern languages make question mark a metacharacter and
allow you to write `a?` for "at most one a".

In [11]:
(regexp-match #px"ca?dr" "cadr")

'("cadr")

In [12]:
(regexp-match #px"ca?dr" "caadr")

#f

For "at least one a" modern languages use `a+`, so the plus sign is another
metacharacter. More generally, we often want to specify quantities. For instance, to
match five a ’s extended regular expressions use the curly braces as metacharacters,
with a{5} . Match between two and five of them with a{2,5} and match at least
two with a{2,} . Thus, a+ is shorthand for a{1,} .
As earlier, to match any of these metacharacters you must escape them. For
instance, To be or not to be\? matches the famous question.

In [ ]:
(regexp-match #px"ca+dr" "caaadr")

More generally, we often want to specify quantities. For instance, to
match five a ’s extended regular expressions use the curly braces as metacharacters,
with a{5} . Match between two and five of them with a{2,5} and match at least
two with a{2,} . Thus, a+ is shorthand for a{1,} .

In [13]:
(regexp-match #px"ca{2,5}dr" "caaadr")

'("caaadr")

In [14]:
(regexp-match #px"ca{2,5}dr" "caaaaaadr")

#f

As earlier, to match any of these metacharacters you must escape them. For
instance, To be or not to be\? matches the famous question.